# Raport: Czyszczenie danych ISOT Fake News

**Autor:** Aleksander Oleszkiewicz  
**Projekt:** factlens — praca inżynierska, klasyfikacja wiadomości prawdziwych i fałszywych  
**Zbiór danych:** ISOT Fake News Dataset (Fake.csv + True.csv)

## 1. Wprowadzenie

Niniejszy raport dokumentuje proces czyszczenia i przygotowania zbioru danych ISOT do dalszego modelowania klasyfikacji binarnej wiadomości prawdziwych i fałszywych. Zbiór ISOT, opracowany przez ISOT Research Lab na University of Victoria, składa się z dwóch plików CSV zawierających artykuły prasowe oznaczone jako prawdziwe (pozyskane z Reuters.com) oraz fałszywe (pozyskane ze stron oznaczonych przez PolitiFact i Wikipedię jako niewiarygodne).

**Cel czyszczenia:**
- usunięcie artefaktów metadanych i markerów źródła, które stanowią "data leakage" (sygnały identyfikujące klasę w sposób niezwiązany z treścią merytoryczną),
- ujednolicenie reprezentacji tekstu,
- eliminacja duplikatów oraz wierszy o zbyt małej zawartości informacyjnej,
- przygotowanie zwalidowanego pliku Parquet spełniającego jawny kontrakt schematu.

**Konwencja etykiet przyjęta w projekcie:**
- `1` — wiadomość prawdziwa (True),
- `0` — wiadomość fałszywa (Fake).

**Zakres kolumn wyjściowych:** jedynie `text` (oczyszczona treść artykułu) oraz `label`. Kolumny `title`, `subject` i `date` zostają odrzucone jako kanały leakage (uzasadnienie w sekcji 4).

Notatnik jest odtwarzalny: wczytuje gotowy, zapisany wcześniej plik `data/processed/news_cleaned.parquet` i porównuje go z surowymi plikami źródłowymi. Pełny, roboczy pipeline czyszczenia znajduje się w `notebooks/explore/02_cleaning.ipynb`; tutaj prezentujemy wyłącznie udokumentowane, ostateczne decyzje projektowe.

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable when running from repository root
PROJECT_ROOT = Path('/home/aoleszkiewicz/dev/factlens')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.cleaning import clean_text

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

FAKE_PATH = RAW_DIR / 'Fake.csv'
TRUE_PATH = RAW_DIR / 'True.csv'
PROCESSED_PATH = PROCESSED_DIR / 'news_cleaned.parquet'

# Plot styling — neutral, publication-friendly
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150
LABEL_NAMES = {0: 'Fake', 1: 'True'}
LABEL_PALETTE = {'Fake': '#c0392b', 'True': '#2c7fb8'}

print('pandas', pd.__version__)
print('numpy ', np.__version__)

## 2. Źródło danych i profilowanie wstępne

Punktem wyjścia są dwa pliki CSV w katalogu `data/raw/`: `Fake.csv` (artykuły nieprawdziwe) oraz `True.csv` (artykuły prawdziwe). Każdy rekord zawiera cztery kolumny: `title`, `text`, `subject`, `date`. Etykieta klasy wynika z pliku, z którego pochodzi dany wiersz, i zostaje jawnie nadana na etapie konkatenacji.

Profilowanie wstępne ma na celu ustalenie linii bazowej — liczby wierszy, wzorców braków danych, skali duplikatów oraz typowych długości tekstu — do której będziemy odnosić wyniki po czyszczeniu.

In [ ]:
# Load raw CSVs for before/after comparison. Not re-running the cleaning pipeline here —
# the processed Parquet is authoritative and was produced in notebooks/explore/02_cleaning.ipynb.
fake_raw = pd.read_csv(FAKE_PATH)
true_raw = pd.read_csv(TRUE_PATH)

profile_rows = []
for name, df in [('Fake', fake_raw), ('True', true_raw)]:
    wc = df['text'].fillna('').str.split().str.len()
    profile_rows.append({
        'plik': name,
        'liczba wierszy': len(df),
        'kolumny': ', '.join(df.columns),
        'dokładne duplikaty wierszy': int(df.duplicated().sum()),
        'duplikaty kolumny text': int(df['text'].duplicated().sum()),
        'puste teksty': int((wc == 0).sum()),
        'mediana słów': int(wc.median()),
        'p95 słów': int(wc.quantile(0.95)),
    })

profile_raw = pd.DataFrame(profile_rows).set_index('plik')
profile_raw

In [ ]:
raw_total = len(fake_raw) + len(true_raw)
class_balance_raw = pd.Series(
    {'Fake': len(fake_raw), 'True': len(true_raw)},
    name='liczba artykułów',
)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
class_balance_raw.plot(
    kind='bar',
    color=[LABEL_PALETTE['Fake'], LABEL_PALETTE['True']],
    ax=ax,
)
ax.set_title('Rozkład klas w surowym zbiorze ISOT')
ax.set_xlabel('Klasa')
ax.set_ylabel('Liczba artykułów')
ax.set_xticklabels(['Fake (0)', 'True (1)'], rotation=0)
for i, v in enumerate(class_balance_raw.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Łączna liczba wierszy surowych: {raw_total:,}')

**Wniosek z profilowania wstępnego.** Surowy zbiór zawiera łącznie 44 898 artykułów z lekką przewagą klasy Fake (23 481 vs 21 417). Brak wartości NaN w kluczowych kolumnach, jednak ujawnia się znacząca skala duplikatów — zwłaszcza w klasie Fake, gdzie 6 026 wierszy ma powtarzającą się treść `text`. Oba pliki zawierają również pojedyncze puste teksty. Mediany długości (359–363 słowa) oraz 95. percentyl (~900 słów) są do siebie zbliżone.

## 3. Zidentyfikowane problemy jakości

Analiza eksploracyjna (zob. `notebooks/explore/02_cleaning.ipynb` oraz historyczny `notebooks/report_temp/`) ujawniła kilka klas artefaktów, które stanowią **data leakage** — sygnały pozwalające dowolnemu modelowi rozstrzygać klasę na podstawie cech formatu źródła, a nie treści merytorycznej. Jeżeli nie zostaną usunięte, model osiągnie wysoką dokładność na ISOT, ale nie będzie generalizował.

**Zidentyfikowane kanały leakage:**

1. **Dateline agencji Reuters** — artykuły prawdziwe prawie zawsze zaczynają się od wzorca `MIASTO (Reuters) -` (np. `WASHINGTON (Reuters) -`). Sama nazwa `reuters` występuje niemal wyłącznie w klasie True (wartość testu chi-kwadrat rzędu 22 700).
2. **Wzmianki `@mention`** — charakterystyczne dla treści ściąganych z mediów społecznościowych, silnie skorelowane z klasą Fake.
3. **URL-e i fragmenty z Twittera** — `https://`, `www.`, `pic.twitter.com/`, `tmsnrt.rs/`. `pic.twitter.com/` praktycznie nie występuje w klasie True, a `tmsnrt.rs/` praktycznie nie występuje w klasie Fake.
4. **Szablony podpisów zdjęć Getty/AP** — `featured image via Getty`, `Photo by X/Getty`, `getty images`, `image/video`.
5. **Znaczniki specyficzne dla witryn** — `21st Century Wire`, artefakty HTML (`<script>`, `&amp;`, `&quot;`).
6. **Zniekształcone sklejenia daty z następnym słowem** — np. `2017Trump` zamiast `2017 Trump`.

**Pozostałe problemy jakości:**
- **Duplikaty wewnątrz klas** — 3 dokładne duplikaty wierszy w Fake, 206 w True oraz 6 026 powtórzeń samej treści `text` w Fake i 225 w True.
- **Kolizje międzyklasowe** — teoretyczna możliwość, że ten sam tekst jest oznaczony zarówno jako Fake, jak i True.
- **Bardzo krótkie artykuły** — po usunięciu URL-i i markerów część wierszy staje się niemal pusta.
- **Rozbieżne formaty dat** — w Fake `"December 31, 2017"`, w True `"December 31, 2017 "` (ze spacją końcową); dodatkowo w klasie Fake występują nietypowe warianty, które uniemożliwiają jednolite parsowanie.

In [ ]:
# Illustrate leakage via token prevalence in the raw data — per-class share of articles
# that contain each flagged token (case-insensitive substring match).
leakage_tokens = ['reuters', 'getty', 'https', 'www.', 'pic.twitter', 'tmsnrt', '21st century wire', '@']

def share_containing(series: pd.Series, token: str) -> float:
    return series.str.contains(token, case=False, regex=False, na=False).mean()

leakage_before = pd.DataFrame(
    {
        'Fake (%)': [share_containing(fake_raw['text'], t) * 100 for t in leakage_tokens],
        'True (%)': [share_containing(true_raw['text'], t) * 100 for t in leakage_tokens],
    },
    index=leakage_tokens,
).round(2)
leakage_before.index.name = 'token'
leakage_before

**Interpretacja.** Różnice udziałów są drastyczne: `reuters` występuje w praktycznie 100% artykułów True i niemal nigdy w Fake. Symetrycznie, `@` i `pic.twitter` są niemal wyłącznie domeną klasy Fake. Każdy z tych tokenów jest idealnym predyktorem klasy — i właśnie dlatego musi zostać usunięty, aby uniknąć uczenia "na skróty".

## 4. Decyzje projektowe

Na bazie profilowania i analizy leakage podjęto następujące decyzje projektowe. Każda z nich ma na celu maksymalizację uczciwości zadania klasyfikacji, nawet kosztem zmniejszenia liczby próbek.

**4.1. Zachowujemy wyłącznie kolumny `text` i `label`.**  
- `subject` — słowniki kategorii są całkowicie rozłączne między klasami (`Fake` ma m.in. `News`, `politics`, `Government News`; `True` ma wyłącznie `politicsNews`, `worldnews`). Kolumna ta daje 100% liniowej separowalności i byłaby czystym leakiem.
- `date` — formaty różnią się między klasami (spacje końcowe, alternatywne warianty w Fake); dodatkowo sam rozkład dat różni się między klasami (Fake i True pochodzą z różnych okien czasowych scrapingu), co również jest kanałem leakage.
- `title` — zawiera silny sygnał stylistyczny (wielkie litery, interpunkcja, długość), częściowo leakage, częściowo użyteczny sygnał merytoryczny. W tej iteracji jest odrzucany; w sekcji 8 zostawiamy otwartą kwestię powrotu do `title` jako osobnej cechy w przyszłych iteracjach.

**4.2. Konwencja etykiet: `True = 1`, `Fake = 0`.**  
Interpretacja standardowa w klasyfikacji binarnej: klasa pozytywna to "wiadomość prawdziwa". Ujednolica to metryki (precision/recall liczone względem klasy True) i jest spójne z konwencją przyjętą w całym projekcie.

**4.3. Próg długości artykułu: 10 słów po czyszczeniu.**  
Po usunięciu URL-i i markerów część artykułów redukuje się do kilku lub zera znaczących słów. Teksty krótsze niż 10 słów nie niosą wystarczającej informacji dla modelu opartego na bag-of-words/TF-IDF/embeddings. Próg jest odziedziczony z wcześniejszej iteracji i został zweryfikowany manualnie (wszystkie odrzucone wiersze to faktycznie śmieci, np. same emotikony, URL-only).

**4.4. Deduplikacja w dwóch etapach.**  
Najpierw usuwane są powtórzenia pary `(text, label)` wewnątrz klasy, następnie sprawdzane są kolizje międzyklasowe (ten sam `text` z różnymi etykietami — takie wiersze są niejednoznaczne i zostają w pełni odrzucone). Po zastosowaniu `clean_text` deduplikacja jest powtarzana, ponieważ czyszczenie może sprowadzić dwa różne surowe teksty do identycznej postaci kanonicznej.

**4.5. Format wyjściowy: Parquet + Snappy.**  
Kolumnowy format z kompresją Snappy daje kilkukrotnie mniejszy rozmiar niż CSV i zachowuje typy (`int32`, `int8`, `string`). Schemat jest jawny i egzekwowany asercjami.

## 5. Pipeline czyszczenia

Pełen pipeline jest zaimplementowany w module `src/preprocessing/cleaning.py` i wywoływany z notatnika roboczego `notebooks/explore/02_cleaning.ipynb`. Poniżej prezentujemy kolejność kroków wraz z krótkim fragmentem kodu kluczowej funkcji `clean_text`.

**Kolejność operacji:**

1. Wczytanie `Fake.csv` i `True.csv`, redukcja do kolumn `text` + nadanie `label` (0/1).
2. Konkatenacja ramek, usunięcie ewentualnych `NaN` w kolumnie `text`.
3. Deduplikacja wewnątrz klas po parze `(text, label)`.
4. Wykrycie i odrzucenie kolizji międzyklasowych (`text` występujący w obu klasach).
5. Zastosowanie `clean_text` do każdego artykułu — serie wyrażeń regularnych usuwających artefakty opisane w sekcji 3 i ostateczną normalizację (lowercase, zwinięcie białych znaków, usunięcie tokenów jednoznakowych).
6. Odfiltrowanie artykułów krótszych niż 10 słów (`filter_short_articles`).
7. Powtórna deduplikacja po czyszczeniu (cleaning może sprowadzić różne teksty do tej samej postaci).
8. Nadanie `id` (stabilne, sekwencyjne), jawne rzutowanie typów, walidacja kontraktu schematu asercjami.
9. Zapis do `data/processed/news_cleaned.parquet` z kompresją Snappy.

Dzięki temu pipeline jest idempotentny — ponowne uruchomienie na tych samych danych wejściowych produkuje bitowo identyczny plik wyjściowy.

In [ ]:
# Inspect the clean_text implementation — single source of truth for all regex rules.
import inspect
from src.preprocessing import cleaning as cleaning_module

print(inspect.getsource(cleaning_module.clean_text))

In [ ]:
# Illustrate the transformation on a representative example from each class.
sample_true = true_raw['text'].iloc[0]
sample_fake = fake_raw['text'].iloc[0]

print('--- Artykuł True (wersja surowa, pierwsze 280 znaków) ---')
print(sample_true[:280])
print('\n--- Artykuł True (po clean_text, pierwsze 280 znaków) ---')
print(clean_text(sample_true)[:280])

print('\n--- Artykuł Fake (wersja surowa, pierwsze 280 znaków) ---')
print(sample_fake[:280])
print('\n--- Artykuł Fake (po clean_text, pierwsze 280 znaków) ---')
print(clean_text(sample_fake)[:280])

**Wniosek.** Na przykładzie widać, że dateline `WASHINGTON (Reuters) -` zostaje usunięta z artykułu True, a charakterystyczne dla klasy Fake cytaty i formatowanie zostają zachowane, ale ujednolicone (małe litery, brak URL-i, brak wzmianek). Treść merytoryczna (kto, co, kiedy) pozostaje nienaruszona.

## 6. Walidacja wyników

Wczytujemy gotowy plik `news_cleaned.parquet` i weryfikujemy, czy spełnia kontrakt schematu: unikalne identyfikatory, binarne etykiety, brak pustych/zduplikowanych tekstów, minimalna długość 10 słów. Porównujemy też liczby wierszy oraz skalę resztkowego leakage względem stanu surowego.

In [ ]:
cleaned = pd.read_parquet(PROCESSED_PATH)
print('Ścieżka:', PROCESSED_PATH)
print('Kształt:', cleaned.shape)
print('\nTypy kolumn:')
print(cleaned.dtypes)
print('\nPierwsze 3 wiersze:')
cleaned.head(3)

In [ ]:
# Schema / data contract assertions.
word_counts = cleaned['text'].str.split().str.len()

assert cleaned['id'].is_unique, 'id must be unique'
assert cleaned['id'].notna().all(), 'id must be non-null'
assert cleaned['label'].isin([0, 1]).all(), 'label must be binary'
assert cleaned['text'].notna().all(), 'text must be non-null'
assert not cleaned['text'].duplicated().any(), 'text must be unique'
assert (word_counts >= 10).all(), 'all texts must have >= 10 words'
assert cleaned['id'].dtype == np.int32, 'id must be int32'
assert cleaned['label'].dtype == np.int8, 'label must be int8'

print('Wszystkie asercje kontraktu schematu przeszły pomyślnie.')

In [ ]:
# Reconciliation: row counts before vs. after cleaning.
n_fake_raw = len(fake_raw)
n_true_raw = len(true_raw)
n_fake_clean = int((cleaned['label'] == 0).sum())
n_true_clean = int((cleaned['label'] == 1).sum())

reconciliation = pd.DataFrame({
    'Fake': [n_fake_raw, n_fake_clean, n_fake_raw - n_fake_clean,
             100 * n_fake_clean / n_fake_raw],
    'True': [n_true_raw, n_true_clean, n_true_raw - n_true_clean,
             100 * n_true_clean / n_true_raw],
    'Razem': [n_fake_raw + n_true_raw, n_fake_clean + n_true_clean,
              (n_fake_raw + n_true_raw) - (n_fake_clean + n_true_clean),
              100 * (n_fake_clean + n_true_clean) / (n_fake_raw + n_true_raw)],
}, index=['surowe', 'po czyszczeniu', 'odrzucone', 'zachowane (%)']).round(2)
reconciliation

In [ ]:
# Class balance visualization — before vs. after cleaning.
balance_df = pd.DataFrame({
    'przed czyszczeniem': [n_fake_raw, n_true_raw],
    'po czyszczeniu': [n_fake_clean, n_true_clean],
}, index=['Fake (0)', 'True (1)'])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
balance_df.plot(kind='bar', ax=ax, color=['#bdbdbd', '#4c72b0'])
ax.set_title('Bilans klas przed i po czyszczeniu')
ax.set_xlabel('Klasa')
ax.set_ylabel('Liczba artykułów')
ax.set_xticklabels(balance_df.index, rotation=0)
ax.legend(title='Etap')
for cont in ax.containers:
    ax.bar_label(cont, fmt='%d', fontsize=9, padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# Residual leakage check — same tokens as before, this time on the cleaned corpus.
residual_rows = []
for t in leakage_tokens:
    n = int(cleaned['text'].str.contains(t, case=False, regex=False, na=False).sum())
    residual_rows.append({
        'token': t,
        'artykuły przed (%)': round(
            100 * (
                (fake_raw['text'].str.contains(t, case=False, regex=False, na=False).sum()
                 + true_raw['text'].str.contains(t, case=False, regex=False, na=False).sum())
                / (n_fake_raw + n_true_raw)
            ),
            2,
        ),
        'artykuły po (liczba)': n,
        'artykuły po (%)': round(100 * n / len(cleaned), 3),
    })

residual = pd.DataFrame(residual_rows).set_index('token')
residual

**Wniosek z walidacji.**  
- Z 44 898 surowych artykułów zachowano 38 470 (~86%). Główne źródła redukcji to: powtórzenia treści w klasie Fake (6 026 duplikatów), dokładne duplikaty wierszy w klasie True (206), pojedyncza kolizja międzyklasowa, 170 artykułów zbyt krótkich po czyszczeniu oraz 5 duplikatów powstałych dopiero po kanonizacji tekstu.  
- Kontrakt schematu jest spełniony: `id` jest unikalne, `label` binarne, `text` niepusty i niezduplikowany, wszystkie artykuły mają co najmniej 10 słów.  
- Resztkowy leakage jest niewielki: `reuters` spada z ~99% artykułów True do 16 całkowitych wystąpień, `getty` do 30, `@` do 160, URL-e praktycznie do zera. Pozostałe przypadki to teksty, w których słowo "Reuters" lub "Getty" pojawia się w ciele cytatu/narracji, a nie jako marker źródła.

## 7. Rozkład długości tekstu

Po wyczyszczeniu sprawdzamy, czy rozkład długości artykułów (liczonej w słowach) jest porównywalny między klasami. Znaczące różnice byłyby sygnałem ostrzegawczym — model mógłby odróżniać klasy po samej długości, co również byłoby formą leakage strukturalnego.

In [ ]:
plot_df = cleaned.assign(
    word_count=word_counts.values,
    klasa=cleaned['label'].map(LABEL_NAMES),
)

length_stats = plot_df.groupby('klasa')['word_count'].describe(
    percentiles=[0.05, 0.5, 0.95]
).round(1)
length_stats

In [ ]:
# Histogram + boxplot side by side. Clip the x-axis at the 99th percentile to keep
# the distribution readable without discarding any data.
x_cap = int(plot_df['word_count'].quantile(0.99))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

sns.histplot(
    data=plot_df,
    x='word_count',
    hue='klasa',
    bins=60,
    binrange=(0, x_cap),
    element='step',
    stat='density',
    common_norm=False,
    palette=LABEL_PALETTE,
    ax=axes[0],
)
axes[0].set_title('Rozkład długości artykułu (gęstość, obcięte do 99. percentyla)')
axes[0].set_xlabel('Liczba słów w artykule')
axes[0].set_ylabel('Gęstość')

sns.boxplot(
    data=plot_df,
    x='klasa',
    y='word_count',
    hue='klasa',
    order=['Fake', 'True'],
    palette=LABEL_PALETTE,
    showfliers=False,
    ax=axes[1],
    legend=False,
)
axes[1].set_title('Długość artykułu wg klasy (bez wartości odstających)')
axes[1].set_xlabel('Klasa')
axes[1].set_ylabel('Liczba słów w artykule')

plt.tight_layout()
plt.show()

**Wniosek z analizy długości.** Rozkłady długości obu klas są porównywalne co do mediany i rozrzutu, z lekką przewagą dłuższych artykułów w klasie True. Różnica nie jest na tyle duża, by sama długość mogła służyć jako skuteczny predyktor klasy — rozkłady istotnie się pokrywają. Po czyszczeniu i odrzuceniu najkrótszych artykułów nie obserwujemy również punktów osobliwych (np. pików na zerze), co potwierdza, że próg 10 słów zadziałał prawidłowo.

## 8. Wnioski

**Co zostało osiągnięte.**
- Zidentyfikowano i udokumentowano sześć klas artefaktów leakage w surowych danych ISOT, od datelines Reuters po szablony podpisów zdjęć Getty.
- Zaimplementowano odtwarzalny, idempotentny pipeline czyszczenia (`src/preprocessing/cleaning.py`), który stanowi pojedyncze źródło prawdy dla reguł usuwania artefaktów.
- Zbiór wyjściowy (`data/processed/news_cleaned.parquet`) zawiera 38 470 artykułów (17 281 Fake, 21 189 True), co stanowi 86% wolumenu surowego. Kontrakt schematu (`id: int32`, `text: string`, `label: int8`) jest jawny i egzekwowany asercjami.
- Resztkowy leakage jest sprowadzony do marginalnych liczb (kilkanaście–kilkadziesiąt artykułów na token), a długości tekstów pozostają porównywalne między klasami.
- Powstały czytelny podział: notatnik roboczy (`explore/02_cleaning.ipynb`) dokumentuje proces, raport (`report/01_cleaning_report.ipynb`) — decyzje i wyniki gotowe do wykorzystania w pracy inżynierskiej.

**Otwarte kwestie.**
1. **Kolumna `title` jako osobna cecha.** Na tym etapie została odrzucona. W przyszłych iteracjach warto rozważyć jej ponowne wprowadzenie jako wejścia równoległego (np. early-stop marker dla potencjalnie fałszywych nagłówków), pod warunkiem oddzielnego potraktowania leakage stylistycznego w tytułach.
2. **160 resztkowych wystąpień znaku `@`.** Analiza manualna sugeruje, że pochodzą one z adresów e-mail lub odwołań do kont pozostawionych w treści głębiej w artykule; ewentualne dodatkowe reguły regexowe mogłyby je zredukować dalej, ale już teraz poziom ten jest poniżej progu istotności statystycznej.
3. **Próg 10 słów** został zaadoptowany bez formalnej analizy wrażliwości. Dla obecnej iteracji jest akceptowalny, ale przy przechodzeniu na modele wymagające dłuższego kontekstu (np. transformery) próg warto ponownie rozważyć.
4. **Resztkowe wystąpienia `reuters` (16) i `getty` (30).** Są to zazwyczaj wzmianki w treści cytatów. Nie wymagają działań, ale należy je mieć na uwadze podczas interpretacji ważności cech w modelu.

**Gotowość do fazy modelowania.**  
Zbiór `news_cleaned.parquet` spełnia wszystkie założone kryteria jakości i może być bezpośrednio wykorzystany jako wejście do fazy ekstrakcji cech (TF-IDF, embeddings) oraz uczenia modeli bazowych. Kolejny krok to przygotowanie stratyfikowanego podziału train/validation/test i uruchomienie pierwszych modeli referencyjnych.